# A - Title Pattern Matching

In [ ]:
import sys
import os

# Add the project root directory to sys.path so that the src package can be imported 
project_root = os.path.abspath("../../")  # Adjust the relative path as needed
if project_root not in sys.path:
    sys.path.append(project_root)

### Load CRS

Download manually from OECD data explorer (searching for "CRS" -> "CRS: Creditor Reporting System (flows)") and save as `data/raw/CRS.parquet`.

In [ ]:
import pandas as pd

# Load CRS data from parquet file
crs = pd.read_parquet("../../data/raw/CRS.parquet")

Optional: Download CRS parquet file with download functions

In [ ]:
# # Import the necessary functions from src
# from src.download_crs import get_full_crs_parquet_url, download_crs_parquet

# # Get the full CRS parquet URL
# file_url = get_full_crs_parquet_url()

# # Download the CRS data
# crs = download_crs_parquet(file_url)

# # Save raw CRS to parquet
# crs.to_parquet("../../data/raw/CRS.parquet")

In [ ]:
# Reduce crs to only the columns we need
crs = crs[['project_title', 'short_description', 'long_description']]

# Only keep rows with project_title that are unique and keep the first occurrence
crs = crs.drop_duplicates(subset=['project_title'], keep='first')

# Remove row with NaN values in the project_title column (drop_dupicates keeps one None value row)
crs = crs[crs['project_title'].notna()]

### Detect Keywords 

In [ ]:
# Load text processing functions
from src.text_processing import normalize_str, detect_language, lemmatize_str, lemmatize_batch, detect_keywords, detect_acronyms, process_keywords

In [ ]:
# Load keywords from Excel file
keywords_file_path = "../../data/keywords/review/keyword_review_modernization.xlsx"
stat_keywords = pd.read_excel(keywords_file_path, sheet_name="statistics")
stat_acronyms = pd.read_excel(keywords_file_path, sheet_name="statistics acronyms")
stat_blacklist = pd.read_excel(keywords_file_path, sheet_name="stat blacklist")
gender_keywords = pd.read_excel(keywords_file_path, sheet_name="gender")
gender_acronyms = pd.read_excel(keywords_file_path, sheet_name="gender acronyms")

# Drop id column for all keywords with loop (id consistent across languages and can be used to group keywords from different languages, omitted in the following)
for df in [stat_keywords, stat_acronyms, stat_blacklist, 
           gender_keywords, gender_acronyms]:
    if 'id' in df.columns:
        df.drop(columns=['id'], inplace=True)
    
    print(f"{df.columns.tolist()}")


# Process keywords
stat_keywords = process_keywords(stat_keywords, remove_stopwords=True)
gender_keywords = process_keywords(gender_keywords, remove_stopwords=True)
stat_blacklist = process_keywords(stat_blacklist, remove_stopwords=True)

# Lowercase all strings in the acronyms dataframes
stat_acronyms = stat_acronyms.map(lambda x: x.lower() if isinstance(x, str) else x)
gender_acronyms = gender_acronyms.map(lambda x: x.lower() if isinstance(x, str) else x)

In [ ]:
# Normalize and detect title language
crs['normalized_title'] = crs['project_title'].apply(normalize_str)
crs['language'] = crs['normalized_title'].apply(detect_language)

In [ ]:
# Lemmatize in batches (for entire CRS dataset ~23min on Intel i7 Gen 11)
for lang in crs['language'].unique():
    # Filter the DataFrame for the current language
    lang_df = crs[crs['language'] == lang]

    if 'lemmatized_title' not in crs.columns:
        crs['lemmatized_title'] = None  # Initialize the column if it doesn't exist
    crs['lemmatized_title'] = crs['lemmatized_title'].astype(object)
    
    # Process the batch and update the original DataFrame
    crs.loc[lang_df.index, 'lemmatized_title'] = lemmatize_batch(lang_df['normalized_title'].tolist(), lang, batch_size=1000, remove_stopwords=True)

    # Lowercase lemmatized titles in case spacy lemmatizes to uppercase
    crs['lemmatized_title'] = crs['lemmatized_title'].str.lower()

In [ ]:
# Save after lemmatization 
crs.to_feather("../../data/processed/title_matched/crs_lemmatized_titles_wo_stopwords.feather")

# Load the lemmatized CRS data from saved checkpoint
# crs = pd.read_feather("../../data/processed//title_matched/crs_lemmatized_titles_wo_stopwords.feather")

In [ ]:
# Detect keywords in the lemmatized title
crs['stat_keywords'] = crs.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], stat_keywords), axis=1)
crs['stat_blacklist'] = crs.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], stat_blacklist), axis=1)
crs['gen_keywords'] = crs.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], gender_keywords), axis=1)

# Detect acronyms in the normalized title
crs['stat_acronyms'] = crs.apply(lambda row: detect_acronyms(row['normalized_title'], row['language'], stat_acronyms), axis=1)
crs['gen_acronyms'] = crs.apply(lambda row: detect_acronyms(row['normalized_title'], row['language'], gender_acronyms), axis=1)

In [ ]:
# Save result
crs.to_feather("../../data/processed/title_matched/crs_titles_matched_wo_stopwords.feather")

In [ ]:
# Reduce crs to only rows with keywords and acronyms detected for manual inspection 
crs_reduced = crs[
    (crs['stat_keywords'].notna()) |
    (crs['stat_blacklist'].notna()) |
    (crs['gen_keywords'].notna()) |
    (crs['stat_acronyms'].notna()) |
    (crs['gen_acronyms'].notna()) 
]

# To xlsx 
crs_reduced.to_excel("../../data/processed/crs_titles_matched_wo_stopwords.xlsx", index=False)

### Create prediction sets for text mining 